# Setup & Imports

In [1]:
import os
import random
import numpy as np
import torch
import torchaudio
import librosa
from pathlib import Path
from datasets import load_from_disk, Dataset, DatasetDict, Audio
from typing import Optional
from tqdm.auto import tqdm

# Load Data

In [2]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH       = PROJECT_ROOT / 'data' / 'synthetic' / 'v3'
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / 'data' / 'synthetic' / 'audio'
TEANGLANN_AUDIO_DIR = PROJECT_ROOT / 'data' / 'teanglann' / 'wav_files'
OUTPUT_PATH        = PROJECT_ROOT / 'data' / 'synthetic' / 'unpaired_training_set'

In [3]:
paired_ds = load_from_disk(str(DATASET_PATH))
print(paired_ds)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Dataset({
    features: ['audio', 'phonetic', 'English ASR transcriptions', 'synthetic_audio_path'],
    num_rows: 19136
})


# Inspect Data

In [4]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'kʌɹəbʌd',
 'synthetic_audio_path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3'}

# Prepare Data

First let's load the audio into the dataset so the audio is presented in the same way

In [5]:
from datasets import Audio

paired_ds = paired_ds.cast_column("synthetic_audio_path", Audio(sampling_rate=16000))
paired_ds = paired_ds.rename_column("synthetic_audio_path", "synthetic_audio")

In [6]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'kʌɹəbʌd',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3',
  'array': array([ 7.99360578e-15, -8.88178420e-15,  9.76996262e-15, ...,
          1.27620979e-07, -3.58589034e-07, -3.13843884e-09], shape=(15360,)),
  'sampling_rate': 16000}}

forgot to normalize transcriptions to same wav2vec2-friendly format as the teanglann scrapings. better for spaces between characters to keep diacritics together with their phones

In [7]:
import json


SYNTH_VOCAB = {' ': 0, 'aɪ': 1, 'aʊ': 2, 'b': 3, 'd': 4, 'eɪ': 5, 'f': 6, 'g': 7,
               'h': 8, 'iː': 9, 'j': 10, 'k': 11, 'l': 12, 'l̩': 13, 'm': 14, 'm̩': 15,
               'n': 16, 'n̩': 17, 'oʊ': 18, 'p': 19, 's': 20, 't': 21, 'uː': 22, 'v': 23,
               'w': 24, 'z': 25, 'æ': 26, 'ð': 27, 'ŋ': 28, 'ŋ̍': 29, 'ɑː': 30, 'ɔː': 31,
               'ɔɪ': 32, 'ə': 33, 'ɚ': 35, 'ɛ': 36,
               'ɪ': 40, 'ɹ': 41, 'ʃ': 44, 'ʊ': 46, 'ʌ': 47,
               'ʒ': 48, 'ʤ': 50, 'ʧ': 51, 'θ': 52}

# Partition into 2-codepoint and 1-codepoint sets for greedy matching
_PHONES_2 = {k for k in SYNTH_VOCAB if len(k) == 2 and k != ' '}
_PHONES_1 = {k for k in SYNTH_VOCAB if len(k) == 1 and k != ' '}


def _space_separate(ipa: str) -> str:
    """Insert spaces between phones in a collapsed synthetic IPA string."""
    if ' ' in ipa:          # already separated
        return ipa
    phones, i = [], 0
    while i < len(ipa):
        if ipa[i:i+2] in _PHONES_2:
            phones.append(ipa[i:i+2])
            i += 2
        else:
            if ipa[i] in _PHONES_1:
                phones.append(ipa[i])
            i += 1
    return ' '.join(phones)

In [8]:
unknown = set()
for item in paired_ds:
    for phone in _space_separate(item['English ASR transcriptions']).split():
        if phone not in SYNTH_VOCAB:
            unknown.add(phone)

print(unknown if unknown else "All phones covered")

All phones covered


In [9]:
paired_ds = paired_ds.map(
    lambda item: {'English ASR transcriptions': _space_separate(item['English ASR transcriptions'])},
    desc='Normalising synthetic IPA',
)
print('Before:', 'kʌɾəbʌd')
print('After: ', _space_separate('kʌɾəbʌd'))

Normalising synthetic IPA:   0%|          | 0/19136 [00:00<?, ? examples/s]

Before: kʌɾəbʌd
After:  k ʌ ə b ʌ d


In [10]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɹ ə b ʌ d',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3',
  'array': array([ 7.99360578e-15, -8.88178420e-15,  9.76996262e-15, ...,
          1.27620979e-07, -3.58589034e-07, -3.13843884e-09], shape=(15360,)),
  'sampling_rate': 16000}}

## strip diacritics

In [11]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.data_handling.collapse_phonemes import collapse_phones

In [12]:
def normalize_row(row):
    row['phonetic'] = "".join(collapse_phones(row['phonetic']))
    row['English ASR transcriptions'] = "".join(collapse_phones(row['English ASR transcriptions']))
    return row

In [13]:
paired_ds = paired_ds.map(
    lambda row: normalize_row(row),
    load_from_cache_file=False,
    desc="Normalizing phonetic columns"
)

Normalizing phonetic columns:   0%|          | 0/19136 [00:00<?, ? examples/s]

In [14]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɹ ə b ʌ d',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3',
  'array': array([ 7.99360578e-15, -8.88178420e-15,  9.76996262e-15, ...,
          1.27620979e-07, -3.58589034e-07, -3.13843884e-09], shape=(15360,)),
  'sampling_rate': 16000}}

let's check the potentially borked recordings.

In [15]:
UNSUPPORTED_IPA = {
    'ɝ', 'i', 'u', 'ɦ', 'ʔ', 'ʉ', 'ɡ', 'ɨ', 'ɾ'
}

bogus_indices = []

for i, sample in enumerate(paired_ds):
    phones = sample['English ASR transcriptions'].split()
    unsupported = [p for p in phones if p in UNSUPPORTED_IPA]
    if unsupported:
        bogus_indices.append((i, unsupported))

print(f"Potentially bogus recordings: {len(bogus_indices)}")
print(f"That's {len(bogus_indices)/len(paired_ds)*100:.1f}% of the synthetic data")


Potentially bogus recordings: 0
That's 0.0% of the synthetic data


# Build Unpaired Dataset

Split paired rows into two independent halves — Irish audio with Irish phonetics, and synthetic audio with English ASR phonetics — then concatenate and shuffle.

In [16]:
from datasets import concatenate_datasets

irish_half = paired_ds.select_columns(['audio', 'phonetic'])

synth_half = (
    paired_ds
    .select_columns(['synthetic_audio', 'English ASR transcriptions'])
    .rename_columns({
        'synthetic_audio': 'audio',
        'English ASR transcriptions': 'phonetic',
    })
)

unpaired_ds = concatenate_datasets([irish_half, synth_half]).shuffle(seed=42)
print(unpaired_ds)

Dataset({
    features: ['audio', 'phonetic'],
    num_rows: 38272
})


In [17]:
split1 = unpaired_ds.train_test_split(test_size=0.2, seed=42)
split2 = split1['test'].train_test_split(test_size=0.5, seed=42)

unpaired_ds_dict = DatasetDict({
    'train': split1['train'],
    'validation': split2['train'],
    'test': split2['test'],
})

In [18]:
unpaired_ds_dict.save_to_disk(str(OUTPUT_PATH))
print(f"Saved {len(unpaired_ds_dict)} samples to {OUTPUT_PATH}")

Saving the dataset (0/2 shards):   0%|          | 0/30617 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3827 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3828 [00:00<?, ? examples/s]

Saved 3 samples to /home/peter/Desktop/thesis/ThesisProject/data/synthetic/unpaired_training_set


# Create Vocab

In [19]:
unpaired_phonetics = [phone for x in unpaired_ds for phone in x['phonetic'].split()]

In [20]:
vocab_unpaired = list(set(unpaired_phonetics)) + [' ']

In [21]:
vocab_list = list(set(vocab_unpaired))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

In [22]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

In [23]:
vocab_dict

{'a': 1,
 'ai': 2,
 'au': 3,
 'aɪ': 4,
 'aʊ': 5,
 'aː': 6,
 'b': 7,
 'bʲ': 8,
 'c': 9,
 'd': 10,
 'dʲ': 11,
 'e': 12,
 'eɪ': 13,
 'eː': 14,
 'f': 15,
 'fʲ': 16,
 'g': 17,
 'h': 18,
 'hʲ': 19,
 'i': 20,
 'ia': 21,
 'iː': 22,
 'iˑə': 23,
 'j': 24,
 'k': 25,
 'l': 26,
 'lʲ': 27,
 'm': 28,
 'mʲ': 29,
 'n': 30,
 'nʲ': 31,
 'o': 32,
 'oʊ': 33,
 'oː': 34,
 'p': 35,
 'pʲ': 36,
 's': 37,
 't': 38,
 'tʲ': 39,
 'u': 40,
 'ua': 41,
 'uː': 42,
 'uˑə': 43,
 'v': 44,
 'vʲ': 45,
 'w': 46,
 'x': 47,
 'z': 48,
 'zʲ': 49,
 'æ': 50,
 'ç': 51,
 'ð': 52,
 'ŋ': 53,
 'ɑː': 54,
 'ɒ': 55,
 'ɔɪ': 56,
 'ɔː': 57,
 'ə': 58,
 'ɚ': 59,
 'ɛ': 60,
 'ɟ': 61,
 'ɡ': 62,
 'ɣ': 63,
 'ɪ': 64,
 'ɲ': 65,
 'ɹ': 66,
 'ɾ': 67,
 'ɾʲ': 68,
 'ʃ': 69,
 'ʊ': 70,
 'ʌ': 71,
 'ʒ': 72,
 'ʤ': 73,
 'ʧ': 74,
 'θ': 75,
 '|': 0,
 '[UNK]': 76,
 '[PAD]': 77}

In [24]:
# save vocab.json
import json
with open(str(OUTPUT_PATH / 'vocab.json'), 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)